In [1]:
# Stage 1b: apply gap handling (option A) to the merged Plant data
import numpy as np
import pandas as pd
from pathlib import Path
 
PLANT = 1
OUT = Path(r"C:\Users\VICTUS\Documents\College\Semester 3\Machine Learning\Solar Power Generation Data\processed")  # same folder as Stage 1
 
df = pd.read_csv(OUT / f"plant{PLANT}_merged.csv", index_col="DATE_TIME", parse_dates=True)
full = df["N_INV"].max()

In [2]:
df["BAD_TARGET"] = df["AC_POWER"].isna() | (df["N_INV"] < full)
df.loc[df["BAD_TARGET"], ["AC_POWER", "DC_POWER"]] = np.nan

In [4]:
def fill_short_gaps(s, max_len=4):
    isna = s.isna()
    run_id = (isna != isna.shift()).cumsum()
    run_len = isna.groupby(run_id).transform("sum")
    filled = s.interpolate(method="time", limit_area="inside")
    return s.where(~(isna & (run_len <= max_len)), filled)

wcols = ["AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]
before = df[wcols].isna().sum()
for c in wcols:
    df[c] = fill_short_gaps(df[c])
after = df[wcols].isna().sum()

In [5]:
print("Weather NaNs before -> after short-gap fill:")
print(pd.DataFrame({"before": before, "after": after}))
print(f"\nRows with a usable target: {df['AC_POWER'].notna().sum()} of {len(df)}")
print("\nBad-target rows per day (last 8 days):")
print(df["BAD_TARGET"].groupby(df.index.date).sum().tail(8))
 
day = df["IRRADIATION"] > 0.1
good_day = day & df["AC_POWER"].notna()
print(f"\nGood daytime rows: {good_day.sum()}; of which AC_POWER == 0: "
      f"{(df.loc[good_day, 'AC_POWER'] == 0).sum()}")

Weather NaNs before -> after short-gap fill:
                     before  after
AMBIENT_TEMPERATURE      82     77
MODULE_TEMPERATURE       82     77
IRRADIATION              82     77

Rows with a usable target: 3057 of 3264

Bad-target rows per day (last 8 days):
2020-06-10    0
2020-06-11    0
2020-06-12    0
2020-06-13    0
2020-06-14    0
2020-06-15    0
2020-06-16    0
2020-06-17    2
Name: BAD_TARGET, dtype: int64

Good daytime rows: 1393; of which AC_POWER == 0: 0


In [6]:
df.to_csv(OUT / f"plant{PLANT}_clean.csv")
print(f"\nSaved {OUT / f'plant{PLANT}_clean.csv'}")


Saved C:\Users\VICTUS\Documents\College\Semester 3\Machine Learning\Solar Power Generation Data\processed\plant1_clean.csv
